# AI Usage Protocol Notebook

Use this notebook to document relevant AI/LLM interactions for one milestone.

## Motivation and Relevance

This protocol is intended to support the documentation of AI usage in accordance with current university and NRW-related guidelines requiring transparency of AI-assisted work processes. In particular, students must disclose relevant uses of AI tools, including revisions, translations, text generation, image generation, analysis support, structuring support, or other AI-assisted scientific activities.

This protocol is not intended as a raw chat dump and not as a collection of successful AI interactions only.

Students must document all milestone-relevant AI interactions that substantially influenced scientific reasoning, understanding, decision-making, literature evaluation, argumentation, modeling, implementation, or interpretation — independent of whether the AI output was ultimately accepted, revised, or rejected.

Relevant interactions therefore include:

* accepted outputs,
* partially used outputs,
* rejected outputs,
* hallucinated or misleading outputs,
* exploratory interactions,
* and interactions that triggered validation or correction activities.

The protocol focuses on documenting scientific reasoning and validation processes, not only final integration into the submitted work.

For iterative or multi-step workflows, students should document the interaction as a sequence of connected reasoning episodes. This includes relevant follow-up prompts, refinements, manual modifications, validation loops, and corrections whenever they substantially influenced the scientific reasoning or outcome of the milestone work.
## Workflow

1. Execute the notebook initialization cell and fill in the general metadata fields for the milestone work that become visible afterwards.
2. For each scientifically relevant AI-supported reasoning episode, click **Add interaction**.
3. Document the relevant prompts, follow-up refinements, manual modifications, validation activities, evaluations, and reflections related to this reasoning episode.
4. Repeat this process for all relevant AI-supported reasoning episodes that influenced the milestone work — including accepted, revised, rejected, exploratory, or corrective usages.
5. Click **Save protocol JSON** to generate the machine-readable protocol file and the corresponding BibTeX entry (`.bib`).
6. Submit the completed notebook (`.ipynb`), the generated protocol (`.json`), and the generated BibTeX file (`.bib`) together with the milestone artifact.


In [2]:
import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError as e:
    raise ImportError("ipywidgets is required. Install with: pip install ipywidgets") from e

PROTOCOL_VERSION = "0.2"

verification_levels = [
    "none",
    "plausibility-check",
    "secondary-source-check",
    "primary-source-check",
    "methodological-check",
    "empirical-check",
    "multi-source-validation"
]

evaluation_statuses = [
    "accepted",
    "partially-accepted",
    "rejected",
    "revised",
    "unresolved"
]

usefulness_levels = [
    "low",
    "medium",
    "high"
]

usage_types = [
    "idea-generation",
    "literature-search-support",
    "summarization",
    "terminology-clarification",
    "comparison-structuring",
    "argument-critique",
    "text-revision",
    "translation",
    "code-generation",
    "verification-support",
    "modeling-support",
    "requirements engineering-support",
    "other"
]

metadata = {
    "protocol_version": PROTOCOL_VERSION,
    "student_id": "",
    "course": "",
    "milestone": "",
    "topic": "",
    "entries": []
}

student_id = widgets.Text(description="Student ID:", placeholder="anonymous or matriculation-compatible ID")
course = widgets.Text(description="Course:", placeholder="Course/module name")
milestone = widgets.Text(description="Milestone:", placeholder="e.g. literature-analysis")
topic = widgets.Text(description="Topic:", placeholder="Your seminar/lab topic")

metadata_box = widgets.VBox([
    widgets.HTML("<h3>Protocol Metadata</h3>"),
    student_id,
    course,
    milestone,
    topic
])

entries_box = widgets.VBox([])
status_output = widgets.Output()

def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def make_interaction_widget(parent_id=None):
    entry_id = f"e-{uuid.uuid4().hex[:8]}"

    objective = widgets.Textarea(
        description="Objective:",
        placeholder="What scientific task did you try to solve?",
        layout=widgets.Layout(width="100%", height="70px")
    )

    ai_tool = widgets.Text(description="AI tool:", placeholder="e.g. ChatGPT, Claude, Perplexity")
    ai_model = widgets.Text(description="Model:", placeholder="e.g. GPT-5.5, unknown")

    prompt = widgets.Textarea(
        description="Prompt:",
        placeholder="Paste the relevant prompt here.",
        layout=widgets.Layout(width="100%", height="120px")
    )

    ai_output_summary = widgets.Textarea(
        description="AI output summary:",
        placeholder="Summarize the AI output. Do not paste long raw outputs unless required.",
        layout=widgets.Layout(width="100%", height="100px")
    )

    verification_level = widgets.RadioButtons(
        options=verification_levels,
        description="Verification level:",
        value="plausibility-check"
    )

    verification_actions = widgets.Textarea(
        description="Verification actions:",
        placeholder="What exactly did you check, and against which sources/evidence?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    evaluation_status = widgets.RadioButtons(
        options=evaluation_statuses,
        description="Status:",
        value="partially-accepted"
    )

    usefulness = widgets.RadioButtons(
        options=usefulness_levels,
        description="Usefulness:",
        value="medium"
    )

    issues = widgets.Textarea(
        description="Issues:",
        placeholder="List hallucinations, overgeneralizations, missing assumptions, wrong citations, etc.",
        layout=widgets.Layout(width="100%", height="90px")
    )

    used_in_work = widgets.Checkbox(description="Used in submitted work", value=False)
    section = widgets.Text(description="Section:", placeholder="e.g. Section 2.3")
    usage_type = widgets.Dropdown(options=usage_types, description="Usage type:", value="other")
    direct_text_reused = widgets.Checkbox(description="Direct text reused", value=False)

    reflection = widgets.Textarea(
        description="Reflection:",
        placeholder="What did you learn? How did this affect your understanding or scientific decision?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    remove_button = widgets.Button(description="Remove interaction", button_style="danger")

    box = widgets.VBox([
        widgets.HTML(f"<hr><h3>Interaction {entry_id}</h3>"),
        objective,
        widgets.HBox([ai_tool, ai_model]),
        prompt,
        ai_output_summary,
        widgets.HTML("<b>Verification</b>"),
        verification_level,
        verification_actions,
        widgets.HTML("<b>Evaluation</b>"),
        evaluation_status,
        usefulness,
        issues,
        widgets.HTML("<b>Integration</b>"),
        used_in_work,
        section,
        usage_type,
        direct_text_reused,
        widgets.HTML("<b>Reflection</b>"),
        reflection,
        remove_button
    ])

    box.entry_id = entry_id
    box.parent_id = parent_id
    box.fields = {
        "objective": objective,
        "ai_tool": ai_tool,
        "ai_model": ai_model,
        "prompt": prompt,
        "ai_output_summary": ai_output_summary,
        "verification_level": verification_level,
        "verification_actions": verification_actions,
        "evaluation_status": evaluation_status,
        "usefulness": usefulness,
        "issues": issues,
        "used_in_work": used_in_work,
        "section": section,
        "usage_type": usage_type,
        "direct_text_reused": direct_text_reused,
        "reflection": reflection
    }

    def remove_entry(_):
        children = list(entries_box.children)
        if box in children:
            children.remove(box)
            entries_box.children = tuple(children)

    remove_button.on_click(remove_entry)
    return box

def add_interaction(_):
    entries_box.children = tuple(list(entries_box.children) + [make_interaction_widget()])

def serialize_entry(box):
    f = box.fields

    verification_actions = [
        line.strip() for line in f["verification_actions"].value.splitlines()
        if line.strip()
    ]
    issues = [
        line.strip() for line in f["issues"].value.splitlines()
        if line.strip()
    ]

    return {
        "id": box.entry_id,
        "timestamp": now_iso(),
        "parent_id": box.parent_id,
        "objective": f["objective"].value.strip(),
        "ai_tool": {
            "name": f["ai_tool"].value.strip(),
            "model": f["ai_model"].value.strip()
        },
        "prompt": f["prompt"].value.strip(),
        "ai_output_summary": f["ai_output_summary"].value.strip(),
        "verification": {
            "actions": verification_actions,
            "level": f["verification_level"].value
        },
        "evaluation": {
            "status": f["evaluation_status"].value,
            "issues": issues,
            "usefulness": f["usefulness"].value
        },
        "integration": {
            "used_in_work": bool(f["used_in_work"].value),
            "section": f["section"].value.strip(),
            "usage_type": f["usage_type"].value,
            "direct_text_reused": bool(f["direct_text_reused"].value)
        },
        "reflection": f["reflection"].value.strip()
    }


def safe_bibtex_key(text):
    """Create a BibTeX-safe key component."""
    text = (text or "ai-usage-protocol").lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text or "ai-usage-protocol"

def escape_bibtex(value):
    """Escape a minimal set of characters for BibTeX fields."""
    value = str(value or "")
    return (
        value.replace("\\", "\\textbackslash{}")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("&", "\\&")
             .replace("%", "\\%")
             .replace("$", "\\$")
             .replace("#", "\\#")
             .replace("_", "\\_")
    )

def protocol_to_bibtex(protocol, json_filename=None):
    """
    Convert an AI usage protocol dictionary into a BibTeX entry.

    The generated entry cites the submitted AI usage protocol as a
    documented part of the student's scientific workflow. It does not
    replace the required disclosure of specific AI tools in the protocol.
    """
    exported_at = protocol.get("exported_at") or now_iso()
    year = exported_at[:4]
    student = protocol.get("student_id") or "anonymous"
    course_name = protocol.get("course") or "Course"
    milestone_name = protocol.get("milestone") or "Milestone"
    topic_name = protocol.get("topic") or "AI-assisted scientific work"
    version = protocol.get("protocol_version") or PROTOCOL_VERSION
    entries_count = len(protocol.get("entries", []))
    json_file = json_filename or "ai_usage_protocol.json"

    key_parts = [
        "ai-usage-protocol",
        safe_bibtex_key(student),
        safe_bibtex_key(milestone_name),
        year,
    ]
    bib_key = "-".join([part for part in key_parts if part])

    title = f"AI Usage Protocol for {milestone_name}: {topic_name}"
    note = (
        f"Machine-readable documentation of AI-assisted scientific work; "
        f"course: {course_name}; protocol version: {version}; "
        f"documented reasoning episodes: {entries_count}; "
        f"JSON file: {json_file}"
    )

    return f"""@misc{{{bib_key},
  author = {{{escape_bibtex(student)}}},
  title = {{{escape_bibtex(title)}}},
  year = {{{escape_bibtex(year)}}},
  howpublished = {{{escape_bibtex("Submitted AI usage protocol notebook and JSON file")}}},
  note = {{{escape_bibtex(note)}}}
}}"""


def save_protocol(_):
    metadata["student_id"] = student_id.value.strip()
    metadata["course"] = course.value.strip()
    metadata["milestone"] = milestone.value.strip()
    metadata["topic"] = topic.value.strip()
    metadata["exported_at"] = now_iso()
    metadata["entries"] = [serialize_entry(box) for box in entries_box.children]

    base_filename = f"ai_usage_protocol_{metadata['student_id'] or 'anonymous'}_{metadata['milestone'] or 'milestone'}"
    base_filename = base_filename.replace(" ", "_").replace("/", "-")
    json_filename = f"{base_filename}.json"
    bib_filename = f"{base_filename}.bib"

    Path(json_filename).write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    bibtex_entry = protocol_to_bibtex(metadata, json_filename=json_filename)
    Path(bib_filename).write_text(bibtex_entry + "\n", encoding="utf-8")

    with status_output:
        clear_output()
        print(f"Saved protocol JSON: {json_filename}")
        print(f"Saved BibTeX entry: {bib_filename}")
        print(f"Entries: {len(metadata['entries'])}")
        print()
        print("BibTeX entry:")
        print(bibtex_entry)

add_button = widgets.Button(description="Add interaction", button_style="primary")
save_button = widgets.Button(description="Save protocol JSON", button_style="success")

add_button.on_click(add_interaction)
save_button.on_click(save_protocol)

display(metadata_box)
display(widgets.HBox([add_button, save_button]))
display(entries_box)
display(status_output)


# ------------------------------------------------------------------
# Prefilled protocol for RTS_FINAL
# Review each entry and replace any reconstructed prompt/model detail
# with the exact original information when available.
# ------------------------------------------------------------------
student_id.value = "2219941"
course.value = "Electronic Engineering"
milestone.value = "Final Seminar Report: Literature Review, Experiment, Image Generation, Modeling, and Critical Analysis"
topic.value = "Multi-Core: Global Scheduling"

_prefilled_entries = [{'objective': 'Understand and synthesize the assigned research on multicore global scheduling, including G-EDF, P-EDF, EPOS, LITMUSRT, operating-system overhead, WCET inflation, assumptions, and limitations.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Analyze the assigned paper on implementation and evaluation of global and partitioned scheduling in a real-time operating system. Explain the periodic task model, G-EDF, P-EDF, EPOS versus LITMUSRT, the overhead sources, the WCET inflation formula, runtime complexity, assumptions, limitations, and how the material can be organized in a seminar report.', 'ai_output_summary': 'The AI produced structured explanations of the task model, the two EDF scheduling paradigms, the EPOS and LITMUSRT implementation differences, the six overhead sources, the WCET inflation equations, algorithm pseudocode, trade-offs, scalability, and suggested report organization.', 'verification_level': 'multi-source-validation', 'verification_actions': 'Compared the summary against the assigned Gracioli et al. paper.\nChecked the EDF utilization statement against Liu and Layland.\nChecked the overhead-accounting interpretation against Brandenburg.\nCompared multiprocessor scheduling claims with the cited Goossens, Bertogna, and Bastoni references.\nCorrected statements that were stronger than the evidence supported.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'Some early explanations treated Windows as equivalent to LITMUSRT; this was corrected to a qualitative transfer comparison.\nThe distinction between sufficient and necessary multiprocessor tests required correction.\nDetailed numerical claims required direct citation to the primary paper.', 'used_in_work': True, 'section': 'Sections I–IV: Introduction, mathematical model, algorithms, runtime overhead, and reference evaluation', 'usage_type': 'comparison-structuring', 'direct_text_reused': False, 'reflection': 'I learned that scheduling theory cannot be evaluated independently of the operating-system implementation. I also learned to separate results from the reference paper from conclusions supported only by my own local experiment.'}, {'objective': 'Design, run, troubleshoot, and interpret the Windows/Linux WCET experiment and Linux perf measurements.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Create and explain a reproducible CPU-bound experiment that measures the maximum observed execution time without load and with fifteen background workers. Help run the same test on Windows and Ubuntu, use Linux perf to collect context switches, CPU migrations, cache references, cache misses, cycles, and instructions, and interpret the results carefully.', 'ai_output_summary': 'The AI supplied and explained the Python multiprocessing benchmark, Linux perf commands, troubleshooting steps for perf_event_paranoid and file paths, percentage calculations, a Windows/Linux comparison, and careful terminology for sampled WCET and aggregate system interference.', 'verification_level': 'empirical-check', 'verification_actions': 'Executed the benchmark on the AMD Ryzen 7 7435HS under Windows and Ubuntu.\nRecorded all ten no-load and ten loaded runs on each platform.\nRan perf stat and checked the reported counters.\nRecalculated the WCET increases manually: 87.3% on Windows and 48.5% in the Linux perf run.\nConfirmed that perf counters covered the complete program, including background workers.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'The first interpretation called the difference pure OS overhead; it was revised to combined WCET inflation under load and interference.\nThe first Linux no-load measurement showed a warm-up/outlier effect.\nThe ten-run maximum is a sampled WCET, not an analytical hard-real-time bound.', 'used_in_work': True, 'section': 'Section V, Tables III–IV, and Fig. 1', 'usage_type': 'code-generation', 'direct_text_reused': False, 'reflection': 'I learned the difference between a sampled maximum and a formally bounded WCET. The perf results also showed that system-level counters support an interference explanation but do not isolate individual context-switch, tick, IPI, or CPMD costs.'}, {'objective': 'Implement and verify a small UPPAAL model that transfers the measured WCET values into two-core G-EDF safe and deadline-miss cases.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Using two cores and two simultaneously released tasks under G-EDF, create one UPPAAL XML model containing a safe case and a miss case. Use the measured Linux values T1 = 309 ms and T2 = 458 ms, verify the chosen deadlines, avoid artificial deadlock, and provide the UPPAAL queries and expected verdicts.', 'ai_output_summary': 'The AI produced and iteratively corrected an UPPAAL XML model with safe and miss automata, self-loops in terminal locations, measured timing constants, deadline guards, and nine verification queries. It also explained why G-EDF uses one global queue and why the 400 ms deadline exposes the WCET-inflation effect.', 'verification_level': 'empirical-check', 'verification_actions': 'Opened the XML model in UPPAAL and simulated both cases.\nChecked all verifier queries and recorded the green/true and red/false verdicts.\nConfirmed that A[] !deadlock is true after terminal self-loops were added.\nConfirmed that the final falsified property is expected because 458 ms exceeds the 400 ms deadline.\nChecked that the model represents one global queue rather than two partitioned queues.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'An early model had an artificial final-state deadlock.\nAn early explanation incorrectly referred to two queues; G-EDF requires one global queue.\nThe model is a timing abstraction and not a complete implementation of arbitrary releases, preemptions, migrations, IPIs, EPOS, or LITMUSRT.', 'used_in_work': True, 'section': 'Section VI, Table V, and Fig. 2', 'usage_type': 'modeling-support', 'direct_text_reused': False, 'reflection': 'I learned that UPPAAL proves properties only for the behavior admitted by the model. The model formally shows the deadline consequence of the measured values, but it does not by itself prove the correctness of a complete multicore operating-system scheduler.'}, {'objective': 'Create clear visualizations for the measured WCET comparison and the two-core G-EDF deadline-miss timeline.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Create publication-ready figures for an IEEE report: a comparison of Windows and Linux sampled WCET with and without background load, and a two-core G-EDF timeline showing T1 = 309 ms, T2 = 458 ms, the 400 ms deadline, and the 58 ms miss region. Make the labels readable in a two-column paper.', 'ai_output_summary': 'The AI generated the WCET comparison chart and G-EDF timeline, then regenerated the timeline after label overlap was detected. It also helped package the figure assets correctly so they appeared in the compiled PDF.', 'verification_level': 'methodological-check', 'verification_actions': 'Compared every plotted number with the recorded experiment output.\nChecked that T1 ends at 309 ms, T2 ends at 458 ms, and the deadline is at 400 ms.\nInspected the compiled IEEE PDF for legibility and label overlap.\nReplaced the first unclear timeline with a corrected version.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': 'The first timeline had overlapping labels.\nAn earlier LaTeX build displayed filename placeholders because the figure assets were not embedded correctly.\nThe figures illustrate selected cases and must not be interpreted as a complete scheduler trace.', 'used_in_work': True, 'section': 'Figures 1 and 2', 'usage_type': 'other', 'direct_text_reused': False, 'reflection': 'I learned that visual accuracy includes both correct values and readable presentation. I manually checked the generated figures and requested revisions when the first version was unclear.'}, {'objective': 'Structure, draft, revise, and format the final five-page IEEE seminar report while preserving the verified research and experimental results.', 'ai_tool': 'ChatGPT', 'ai_model': 'GPT-5.5 / GPT-5.6 Thinking (session-dependent)', 'prompt': 'Using the assigned research paper, final presentation, draft document, experiment outputs, Linux perf results, and UPPAAL verification, prepare a five-page IEEE-format report on Multi-Core: Global Scheduling. Improve the structure and language, keep the technical results accurate, add limitations and an AI-use acknowledgment, and fix LaTeX compilation problems.', 'ai_output_summary': 'The AI proposed the report structure, drafted and revised prose, produced IEEE LaTeX, integrated equations, algorithms, tables, figures, citations, limitations, and the acknowledgment, and helped diagnose Unicode, figure-path, and compilation errors.', 'verification_level': 'methodological-check', 'verification_actions': 'Compared the final report against the primary reference and the recorded experiment outputs.\nChecked equations, task parameters, percentages, UPPAAL constants, and query verdicts.\nCompiled the LaTeX source and inspected the complete PDF.\nReviewed citations and corrected unsupported or overly strong statements.\nRetained responsibility for final wording and submission.', 'evaluation_status': 'revised', 'usefulness': 'high', 'issues': "Several generated drafts contained awkward grammar, duplicated phrases, invisible Unicode characters, and the title typo 'Schdeuling'.\nCitation placement and the missing UPPAAL reference required manual review.\nSome AI-generated wording was too polished or generic and required rewriting to reflect the actual work.", 'used_in_work': True, 'section': 'Whole report, especially organization, language revision, LaTeX formatting, and acknowledgment', 'usage_type': 'text-revision', 'direct_text_reused': True, 'reflection': 'I learned to treat AI output as a draft rather than an authority. Technical claims, equations, citations, and experimental values were checked separately. AI improved organization and formatting, but the final scientific decisions and responsibility remained mine.'}, {'objective': 'Obtain an additional language and structure review of the seminar report using Claude.', 'ai_tool': 'Claude', 'ai_model': 'Unknown / not recorded', 'prompt': 'Reconstructed purpose of the interaction: review the seminar report for academic clarity, structure, grammar, and consistency while preserving the experimental results and technical meaning. Replace this text with the exact original prompt if it is still available.', 'ai_output_summary': 'Claude was used as an additional drafting and editing aid. Suggestions concerned wording, organization, and presentation. Only selected suggestions were retained after comparison with the research sources and the measured results.', 'verification_level': 'methodological-check', 'verification_actions': 'Compared suggested changes with the final technical content and original references.\nRejected wording that altered numerical results, assumptions, or the distinction between sampled WCET and analytical WCET.\nManually reviewed all retained language changes.', 'evaluation_status': 'partially-accepted', 'usefulness': 'medium', 'issues': 'The exact Claude model and original prompt were not retained.\nThis entry documents the purpose of the interaction; the reconstructed prompt should be replaced if the original is available.\nLanguage suggestions were not treated as scientific evidence.', 'used_in_work': True, 'section': 'Language and structure review across the report', 'usage_type': 'text-revision', 'direct_text_reused': False, 'reflection': 'Using a second AI tool showed that different tools can suggest different wording, but neither replaces verification against the paper, experiment logs, and UPPAAL results.'}]

_prefilled_boxes = []
for _data in _prefilled_entries:
    _box = make_interaction_widget()
    _f = _box.fields
    _f["objective"].value = _data["objective"]
    _f["ai_tool"].value = _data["ai_tool"]
    _f["ai_model"].value = _data["ai_model"]
    _f["prompt"].value = _data["prompt"]
    _f["ai_output_summary"].value = _data["ai_output_summary"]
    _f["verification_level"].value = _data["verification_level"]
    _f["verification_actions"].value = _data["verification_actions"]
    _f["evaluation_status"].value = _data["evaluation_status"]
    _f["usefulness"].value = _data["usefulness"]
    _f["issues"].value = _data["issues"]
    _f["used_in_work"].value = _data["used_in_work"]
    _f["section"].value = _data["section"]
    _f["usage_type"].value = _data["usage_type"]
    _f["direct_text_reused"].value = _data["direct_text_reused"]
    _f["reflection"].value = _data["reflection"]
    _prefilled_boxes.append(_box)

entries_box.children = tuple(_prefilled_boxes)


VBox()

Output()

## Recommended Use

For every milestone, start from a fresh copy of this notebook template.

Suggested rule for students:

- Do **not** document every trivial AI interaction.
- Do document every interaction that influenced a scientific decision, interpretation, comparison, argument, model, implementation, or submitted text.
- The most important part is not the prompt. The most important part is the **verification and evaluation**.

Suggested submission:

- completed notebook `.ipynb`
- generated protocol `.json`
- generated BibTeX file `.bib`
- milestone artifact, e.g. literature analysis, outline, model, code, or paper draft


## Controlled Vocabulary

### Verification level

- `none`: accepted without checking
- `plausibility-check`: checked only for plausibility
- `secondary-source-check`: checked against tutorials, summaries, lecture material, or secondary sources
- `primary-source-check`: checked against original papers, standards, documentation, or data
- `methodological-check`: checked the reasoning, method, assumptions, or formal correctness
- `empirical-check`: checked by running an experiment, model, tool, test, or implementation
- `multi-source-validation`: checked against multiple independent reliable sources

### Evaluation status

- `accepted`: used essentially as provided
- `partially-accepted`: used selectively
- `rejected`: discarded after evaluation
- `revised`: substantially corrected or rewritten
- `unresolved`: uncertainty remains


## Optional: Convert an Existing Protocol JSON to BibTeX

Use the following cell if you already have a generated protocol `.json` file and need to recreate the corresponding BibTeX entry.


In [3]:
# Optional JSON-to-BibTeX converter
#
# Change this filename if needed, then run the cell.
# The generated .bib file can be included in the references of the scientific work.

json_protocol_file = "ai_usage_protocol_anonymous_milestone.json"

def convert_json_protocol_to_bibtex(json_file):
    json_path = Path(json_file)
    if not json_path.exists():
        raise FileNotFoundError(f"Protocol JSON file not found: {json_file}")

    protocol = json.loads(json_path.read_text(encoding="utf-8"))
    bibtex_entry = protocol_to_bibtex(protocol, json_filename=json_path.name)
    bib_path = json_path.with_suffix(".bib")
    bib_path.write_text(bibtex_entry + "\n", encoding="utf-8")
    print(f"Saved BibTeX entry: {bib_path}")
    print()
    print(bibtex_entry)

# Uncomment the next line after setting json_protocol_file correctly:
# convert_json_protocol_to_bibtex(json_protocol_file)
